# 삼성전자(005930.KS) 2023~2024년 주가 시계열 트렌드 및 변동성 분석

- **분석 기간**: 2023-01-01 ~ 2024-12-31
- **데이터 출처**: Yahoo Finance (`005930.KS`)
- **작성 목적**: 시계열 데이터 정제, 이동평균선/변동성 분석, 시계열 분해 및 예측, 데이터 기반 인사이트 도출

## 1. 환경 설정 및 데이터 로드

In [ ]:
import os
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# 한글 폰트 설정
system = platform.system()
font_name = 'Apple SD Gothic Neo' if system == 'Darwin' else ('Malgun Gothic' if system == 'Windows' else 'NanumGothic')
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=font_name)

# 데이터 로드
csv_path = 'data/samsung_2023_2024.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, parse_dates=['Date'], index_col='Date')
else:
    stock = yf.Ticker('005930.KS')
    df = stock.history(start='2023-01-01', end='2024-12-31', auto_adjust=False)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    os.makedirs('data', exist_ok=True)
    df.to_csv(csv_path)

print('데이터 크기:', df.shape)
df.head()

## 2. 결측치 확인 및 파생 변수(지표) 생성

- 일간 수익률 (`Daily_Return`)
- 20일, 60일, 120일 이동평균선 (`MA20`, `MA60`, `MA120`)
- 20일 롤링 변동성 (연율화, `Volatility_20d`)
- 골든크로스 / 데드크로스 신호 (`Signal`)

In [ ]:
print('결측치 현황:\n', df.isnull().sum())

# 지표 생성
df['Daily_Return'] = df['Close'].pct_change()
df['MA20'] = df['Close'].rolling(window=20).mean()
df['MA60'] = df['Close'].rolling(window=60).mean()
df['MA120'] = df['Close'].rolling(window=120).mean()
df['Volatility_20d'] = df['Daily_Return'].rolling(window=20).std() * np.sqrt(252) * 100

ma_diff = df['MA20'] - df['MA60']
df['Signal'] = 0
df.loc[(ma_diff > 0) & (ma_diff.shift(1) <= 0), 'Signal'] = 1   # Golden Cross
df.loc[(ma_diff < 0) & (ma_diff.shift(1) >= 0), 'Signal'] = -1  # Dead Cross

df.describe()

## 3. 핵심 시계열 시각화

### 3.1 주가 추세 및 거래량

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df.index, df['Close'], label='종가 (KRW)', color='#1f77b4', lw=2)
max_idx, max_val = df['Close'].idxmax(), df['Close'].max()
min_idx, min_val = df['Close'].idxmin(), df['Close'].min()

ax1.scatter([max_idx], [max_val], color='#d62728', s=100, zorder=5, label=f'최고가: {int(max_val):,}원')
ax1.scatter([min_idx], [min_val], color='#2ca02c', s=100, zorder=5, label=f'최저가: {int(min_val):,}원')
ax1.set_title('삼성전자(005930.KS) 2023~2024년 주가 추세 및 변곡점', fontsize=15, fontweight='bold')
ax1.set_ylabel('주가 (원)')
ax1.legend()

colors = ['#2ca02c' if r >= 0 else '#d62728' for r in df['Daily_Return'].fillna(0)]
ax2.bar(df.index, df['Volume'] / 1_000_000, color=colors, alpha=0.7, width=1.2)
ax2.set_ylabel('거래량 (백만 주)')
ax2.set_xlabel('거래 일자')
plt.tight_layout()
plt.show()

### 3.2 이동평균선(20/60/120일) 및 골든·데드크로스

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Close'], label='종가', color='#8c8c8c', alpha=0.6)
plt.plot(df.index, df['MA20'], label='MA 20일', color='#e41a1c', lw=1.8)
plt.plot(df.index, df['MA60'], label='MA 60일', color='#377eb8', lw=1.8)
plt.plot(df.index, df['MA120'], label='MA 120일', color='#4daf4a', lw=1.8, linestyle='--')

gc = df[df['Signal'] == 1]
dc = df[df['Signal'] == -1]
plt.scatter(gc.index, gc['MA20'], color='red', marker='^', s=120, label=f'골든크로스 ({len(gc)}회)')
plt.scatter(dc.index, dc['MA20'], color='blue', marker='v', s=120, label=f'데드크로스 ({len(dc)}회)')

plt.title('중단기 이동평균선 및 크로스 신호', fontsize=15, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

### 3.3 20일 롤링 변동성 추이

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Volatility_20d'], color='#756bb1', lw=2, label='20일 롤링 변동성 (%)')
plt.axhline(df['Volatility_20d'].mean(), color='#e6550d', linestyle='--', label=f'평균 ({df["Volatility_20d"].mean():.1f}%)')
plt.title('삼성전자 20일 롤링 연율화 변동성 (%)', fontsize=14, fontweight='bold')
plt.ylabel('변동성 (%)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. (보너스) 시계열 분해 및 베이스라인 예측

In [ ]:
# 4.1 시계열 성분 분해
daily_ts = df['Close'].asfreq('D').ffill()
decomp = seasonal_decompose(daily_ts, model='additive', period=30)
fig = decomp.plot()
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.show()

# 4.2 지수평활 베이스라인 예측
train = df['Close'].iloc[:-20]
test = df['Close'].iloc[-20:]
fit = ExponentialSmoothing(train.values, trend='add', damped_trend=True).fit()
pred = fit.forecast(20)

plt.figure(figsize=(12, 5))
plt.plot(train.iloc[-40:].index, train.iloc[-40:].values, label='학습 데이터')
plt.plot(test.index, test.values, label='실제값(Test)', color='green', marker='o')
plt.plot(test.index, pred, label='지수평활 예측', color='red', linestyle='--', marker='s')
plt.title('최근 20거래일 베이스라인 예측 vs 실측치')
plt.legend()
plt.tight_layout()
plt.show()